# Getting Started with the ML/AI Pipeline

This notebook demonstrates how to use the ML/AI pipeline template for your projects.

In [ ]:
# Add project root to path
import sys
sys.path.append('..')

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import pipeline modules
from src.utils import get_config
from src.data_engineering import load_data, validate_data, DataPipeline, split_data
from src.data_analysis import analyze_data, visualize_data
from src.ml import MLTrainer, ModelEvaluator

# Display settings
pd.set_option('display.max_columns', None)
%matplotlib inline

## 1. Load Configuration

In [ ]:
# Load configuration
config = get_config('../config/config.yaml')
print(f"Project: {config.get('project.name')}")
print(f"Version: {config.get('project.version')}")

## 2. Load Your Data

**NOTE**: Add your data file to `data/raw/` directory first!

In [ ]:
# Example: Load data (replace with your data file)
# data = load_data('../data/raw/your_data.csv')

# For demo, create sample data
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    random_state=42
)

data = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
data['target'] = y

print(f"Data shape: {data.shape}")
data.head()

## 3. Data Validation

In [ ]:
# Validate data
validation_results = validate_data(data)

print("Validation Results:")
print(f"Missing values: {validation_results['missing_values']['total_missing']}")
print(f"Duplicates: {validation_results['duplicates']['n_duplicates']}")

## 4. Exploratory Data Analysis

In [ ]:
# Run EDA
eda_report = analyze_data(data)

# Display summary statistics
eda_report['statistical_summary']['numeric']

## 5. Data Splitting and Preprocessing

In [ ]:
# Separate features and target
X = data.drop('target', axis=1)
y = data['target']

# Split data
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X, y,
    test_size=0.2,
    val_size=0.1,
    random_state=42
)

print(f"Train size: {X_train.shape}")
print(f"Val size: {X_val.shape}")
print(f"Test size: {X_test.shape}")

In [ ]:
# Preprocess data
pipeline = DataPipeline(config.config)
X_train_processed = pipeline.fit_transform(X_train)
X_val_processed = pipeline.transform(X_val)
X_test_processed = pipeline.transform(X_test)

print(f"Processed train shape: {X_train_processed.shape}")

## 6. Train Machine Learning Models

In [ ]:
# Initialize trainer
trainer = MLTrainer(task_type='classification')

# Train a Random Forest model
model = trainer.train(
    X_train_processed,
    y_train,
    'random_forest',
    params={'n_estimators': 100, 'max_depth': 10, 'random_state': 42}
)

print("Model trained successfully!")

## 7. Model Evaluation

In [ ]:
# Make predictions
y_pred = trainer.predict('random_forest', X_test_processed)
y_pred_proba = trainer.predict_proba('random_forest', X_test_processed)

# Evaluate
evaluator = ModelEvaluator(task_type='classification')
metrics = evaluator.evaluate(
    y_test,
    y_pred,
    model_name='random_forest',
    y_pred_proba=y_pred_proba
)

print("\nEvaluation Metrics:")
for metric, value in metrics.items():
    if isinstance(value, (int, float)):
        print(f"{metric}: {value:.4f}")

In [ ]:
# Plot confusion matrix
evaluator.plot_confusion_matrix(y_test, y_pred, 'random_forest')
plt.show()

## 8. Feature Importance

In [ ]:
# Get feature importances
importances = trainer.get_feature_importance('random_forest')

if importances is not None:
    feature_names = X_train_processed.columns.tolist()
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print("Top 10 Important Features:")
    print(importance_df.head(10))
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.barh(importance_df['feature'][:10], importance_df['importance'][:10])
    plt.xlabel('Importance')
    plt.title('Top 10 Feature Importances')
    plt.gca().invert_yaxis()
    plt.show()

## 9. Save Model

In [ ]:
# Save model
trainer.save_model('random_forest', '../models/saved_models/random_forest.pkl')
print("Model saved!")

## Next Steps

1. Try different models (XGBoost, LightGBM, etc.)
2. Tune hyperparameters using `HyperparameterTuner`
3. Use experiment tracking with MLflow
4. Deploy your model using the FastAPI server
5. Explore deep learning notebooks for neural networks
6. Check RL notebooks for reinforcement learning

See other notebooks in this directory for more examples!